# Stage 3: Structured Streaming Consumer
## ISM 6562 — Final Project — MediStream Telehealth


## Setup

The Spark session needs the Kafka connector package (`spark-sql-kafka-0-10`). Internet is required on first run so Spark can download it from Maven Central; subsequent runs are cached.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

KAFKA_BOOTSTRAP = 'kafka:9093'
TOPIC_IN  = 'session-metrics'
TOPIC_OUT = 'session-alerts'

spark = (SparkSession.builder
    .appName('MediStream-Stage3b-Consumer')
    .master('spark://spark-master:7077')
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0')
    .config('spark.executor.memory', '1g')
    .config('spark.driver.memory', '1g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.streaming.checkpointLocation', '/home/jovyan/data/checkpoints/streaming-consumer')
    .getOrCreate())

print(f'Spark version: {spark.version}')
print(f'Application ID: {spark.sparkContext.applicationId}')

## Input schema and read stream

In [ ]:
metrics_schema = StructType([
    StructField('session_id',          StringType()),
    StructField('appointment_id',      StringType()),
    StructField('video_resolution',    StringType()),
    StructField('audio_quality_score', DoubleType()),
    StructField('latency_ms',          DoubleType()),
    StructField('packet_loss_pct',     DoubleType()),
    StructField('duration_sec',        DoubleType()),
    StructField('device_type',         StringType()),
    StructField('os',                  StringType()),
    StructField('event_time',          StringType()),  # ISO-8601 string, parsed below
])

raw = (spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', KAFKA_BOOTSTRAP)
    .option('subscribe', TOPIC_IN)
    .option('startingOffsets', 'earliest')
    .option('failOnDataLoss', 'false')
    .load())

events = (raw
    .selectExpr('CAST(value AS STRING) AS value', 'timestamp AS kafka_ts')
    .select(F.from_json(F.col('value'), metrics_schema).alias('m'), 'kafka_ts')
    .select('m.*', 'kafka_ts')
    .withColumn('event_time', F.to_timestamp('event_time')))

events.printSchema()

## Windowed aggregation

2-minute sliding window with 30-second slide. The watermark of 2 minutes drops state for windows whose end is more than 2 minutes behind the max observed event time — bounded memory.

In [ ]:
windowed = (events
    .withWatermark('event_time', '2 minutes')
    .groupBy(
        F.window('event_time', '2 minutes', '30 seconds'),
        'session_id', 'appointment_id', 'device_type', 'os')
    .agg(
        F.count('*').alias('sample_count'),
        F.avg('latency_ms').alias('avg_latency_ms'),
        F.avg('packet_loss_pct').alias('avg_packet_loss_pct'),
        F.avg('audio_quality_score').alias('avg_audio_quality_score'),
    )
    .select(
        F.col('window.start').alias('window_start'),
        F.col('window.end').alias('window_end'),
        'session_id', 'appointment_id', 'device_type', 'os',
        'sample_count',
        F.round('avg_latency_ms', 2).alias('avg_latency_ms'),
        F.round('avg_packet_loss_pct', 3).alias('avg_packet_loss_pct'),
        F.round('avg_audio_quality_score', 2).alias('avg_audio_quality_score'),
    ))

windowed.printSchema()

## Threshold tuning

Brief gives baseline thresholds (latency >500ms, packet_loss >5%) for the alert. Pick the actual numbers — tighter = more alerts, fewer false-negatives but more noise; looser = the opposite. Audio quality threshold is your call (codec scores 1–10, mid-scale is ~5).

In [ ]:
# Thresholds match the brief and the batch tables in 02c/02e so batch and
# streaming agree on what 'degraded' means.
LATENCY_ALERT_MS       = 500
PACKET_LOSS_ALERT_PCT  = 5.0
AUDIO_QUALITY_ALERT    = 5

for name, val, kind in [
    ('LATENCY_ALERT_MS',      LATENCY_ALERT_MS,     (int, float)),
    ('PACKET_LOSS_ALERT_PCT', PACKET_LOSS_ALERT_PCT,(int, float)),
    ('AUDIO_QUALITY_ALERT',   AUDIO_QUALITY_ALERT,  (int, float)),
]:
    assert isinstance(val, kind) and val > 0, f'{name} must be a positive number'
print(f'Thresholds - latency>{LATENCY_ALERT_MS}ms, '
      f'packet_loss>{PACKET_LOSS_ALERT_PCT}%, audio<{AUDIO_QUALITY_ALERT}')

## Filter to degraded windows + classify alert type

Each row is a (session × window) bucket. Keep only those where at least one threshold is breached, then label the alert.

In [ ]:
degraded = windowed.filter(
    (F.col('avg_latency_ms') > LATENCY_ALERT_MS) |
    (F.col('avg_packet_loss_pct') > PACKET_LOSS_ALERT_PCT) |
    (F.col('avg_audio_quality_score') < AUDIO_QUALITY_ALERT)
)

# Single most-severe alert label per window. Severity order is
# patient-safety-conservative for telemedicine:
#   low_audio > packet_loss > high_latency
# Clinical reasoning: missing what the patient says (low audio = missed verbal
# symptoms, missed wearable breath/heart sounds) is the worst miss in a
# remote consult. Packet loss is bad but often user-noticed. Latency is
# annoying but the call still functions.
alert_type = (
    F.when(F.col('avg_audio_quality_score') < AUDIO_QUALITY_ALERT, F.lit('low_audio'))
     .when(F.col('avg_packet_loss_pct') > PACKET_LOSS_ALERT_PCT,    F.lit('packet_loss'))
     .when(F.col('avg_latency_ms') > LATENCY_ALERT_MS,              F.lit('high_latency'))
     .otherwise(F.lit('unknown'))
)

labeled = degraded.withColumn('alert_type', alert_type)
labeled.printSchema()

## Enrich with appointment + physician context

Broadcast-join the curated `appointments` dim so each alert carries `physician_id` + `specialty`. Broadcast is fine here — ~550k rows × ~5 cols easily fits in driver memory.

In [ ]:
HDFS_CURATED  = 'hdfs://namenode:9000/medistream/curated/appointments'
LOCAL_CURATED = '/home/jovyan/data/curated/appointments'
try:
    appt_dim = spark.read.parquet(HDFS_CURATED).select(
        'appointment_id', 'physician_id', 'specialty', 'visit_type')
    print(f'Loaded appt dim from HDFS')
except Exception:
    appt_dim = spark.read.parquet(LOCAL_CURATED).select(
        'appointment_id', 'physician_id', 'specialty', 'visit_type')
    print(f'Loaded appt dim from local mount')
print(f'Appt dim rows: {appt_dim.count():,}')

enriched = labeled.join(F.broadcast(appt_dim), on='appointment_id', how='left')
enriched.printSchema()

## Suggest an action

What MediStream should tell the patient/physician *during the call* when an alert fires. Different alert types call for different actions.

In [ ]:
# Short imperative strings the UI surfaces during the call. Mapped 1:1 from
# alert_type - keeps the messaging predictable for users.
suggested_action = (
    F.when(F.col('alert_type') == 'high_latency', F.lit('reduce video resolution to 480p'))
     .when(F.col('alert_type') == 'packet_loss',  F.lit('switch to audio-only mode'))
     .when(F.col('alert_type') == 'low_audio',    F.lit('check headset/microphone connection'))
     .otherwise(F.lit('check connection'))
)

alerts = enriched.withColumn('suggested_action', suggested_action) \n    .withColumn('alert_date', F.to_date('window_end'))
alerts.printSchema()

## Sink — write to both Kafka and HDFS via foreachBatch

`foreachBatch` lets one streaming query emit to two sinks atomically. Each micro-batch is a normal DataFrame, so we can call `.write` twice on it.

In [ ]:
HDFS_ALERTS = 'hdfs://namenode:9000/medistream/analytics/streaming_alerts'
LOCAL_ALERTS = '/home/jovyan/data/analytics/streaming_alerts'
try:
    spark.range(1).write.mode('overwrite').parquet(f'{HDFS_ALERTS}/_probe')
    ALERTS_PATH = HDFS_ALERTS
    print('Alerts -> HDFS')
except Exception:
    ALERTS_PATH = LOCAL_ALERTS
    print('Alerts -> local mount (HDFS unavailable)')

def emit_batch(batch_df, batch_id):
    if batch_df.rdd.isEmpty():
        return
    n = batch_df.count()
    print(f'[batch {batch_id}] emitting {n} alert rows')

    # 1) HDFS — append, partitioned by alert_date
    (batch_df.write
        .mode('append')
        .partitionBy('alert_date')
        .parquet(ALERTS_PATH))

    # 2) Kafka — pack the alert envelope as JSON
    kafka_df = (batch_df
        .withColumn('value', F.to_json(F.struct(*[c for c in batch_df.columns])))
        .withColumn('key', F.col('session_id'))
        .select('key', 'value'))
    (kafka_df.write
        .format('kafka')
        .option('kafka.bootstrap.servers', KAFKA_BOOTSTRAP)
        .option('topic', TOPIC_OUT)
        .save())

query = (alerts.writeStream
    .foreachBatch(emit_batch)
    .outputMode('append')
    .trigger(processingTime='30 seconds')
    .start())

print('Streaming query started. id =', query.id)
print('Run 03a-streaming-producer.ipynb in another window to feed events.')
print('Spark App UI: http://localhost:4040')

In [ ]:
# Block here until you stop the kernel. The producer notebook will publish
# events; this consumer will print one line per micro-batch as alerts emit.
query.awaitTermination()

In [ ]:
# Run this cell to stop the streaming query cleanly
# query.stop()